# 03 - Business relationship model
Turns the verified table structure into a business view: six components, how they interact, and what the data can and cannot support.

The raw CSV files are never modified.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# works whether the notebook is run from the project root or from notebooks/
here = Path.cwd().resolve()
PROJECT_ROOT = here.parent if here.name.lower() == "notebooks" else here
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}. Run notebook 01 first.")

FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
dfs = {name: pd.read_csv(DATA_DIR / f) for name, f in FILES.items()}

for name, df in dfs.items():
    print(f"{name:22} rows={len(df):>9,}  cols={df.shape[1]}")

## 1. Keys and foreign keys
Same checks as notebook 02, repeated here so this notebook stands on its own.

In [ ]:
KEYS = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],   # review_id alone is not unique
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

key_rows = []
for table, keys in KEYS.items():
    df = dfs[table]
    nulls = int(df[keys].isna().any(axis=1).sum())
    dups = int(df.duplicated(subset=keys).sum())
    key_rows.append({"table": table, "key": " + ".join(keys), "null_keys": nulls,
                     "duplicate_keys": dups, "unique": nulls == 0 and dups == 0})
display(pd.DataFrame(key_rows))

In [ ]:
FOREIGN_KEYS = [
    # parent, parent key, child, child key, meaning
    ("customers", "customer_id", "orders", "customer_id", "customer record places order"),
    ("orders", "order_id", "order_items", "order_id", "order contains item lines"),
    ("products", "product_id", "order_items", "product_id", "product appears on item line"),
    ("sellers", "seller_id", "order_items", "seller_id", "seller supplies item line"),
    ("orders", "order_id", "payments", "order_id", "order has payment records"),
    ("orders", "order_id", "reviews", "order_id", "order has review records"),
]

fk_rows = []
for parent, pk, child, fk, meaning in FOREIGN_KEYS:
    orphans = set(dfs[child][fk].dropna()) - set(dfs[parent][pk].dropna())
    fk_rows.append({"parent": f"{parent}.{pk}", "child": f"{child}.{fk}",
                    "unmatched_values": len(orphans), "meaning": meaning})
display(pd.DataFrame(fk_rows))

## 2. Database-level ER diagram
`UNDERLYING_CUSTOMER` is not a CSV table. It is the real person behind a `customer_unique_id`, who can have several customer records.

The translation table is an optional lookup, and geolocation is left out of the core diagram.

```mermaid
erDiagram
    UNDERLYING_CUSTOMER ||--|{ CUSTOMER_RECORD : identifies
    CUSTOMER_RECORD ||--|| ORDER : places
    ORDER ||--|{ ORDER_ITEM : contains
    PRODUCT ||--o{ ORDER_ITEM : appears_in
    SELLER ||--o{ ORDER_ITEM : fulfils
    ORDER ||--o{ PAYMENT : has
    ORDER ||--o{ REVIEW : receives
    PRODUCT_CATEGORY_TRANSLATION o|--o{ PRODUCT : translates

    UNDERLYING_CUSTOMER {
        string customer_unique_id PK
    }
    CUSTOMER_RECORD {
        string customer_id PK
        string customer_unique_id FK
        string customer_city
        string customer_state
    }
    ORDER {
        string order_id PK
        string customer_id FK
        string order_status
        datetime order_purchase_timestamp
    }
    ORDER_ITEM {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        float price
        float freight_value
    }
    PRODUCT {
        string product_id PK
        string product_category_name
        float product_weight_g
    }
    SELLER {
        string seller_id PK
        string seller_city
        string seller_state
    }
    PAYMENT {
        string order_id PK, FK
        int payment_sequential PK
        string payment_type
        float payment_value
    }
    REVIEW {
        string review_id PK
        string order_id PK, FK
        int review_score
    }
    PRODUCT_CATEGORY_TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }
```

Two things to remember:
- One order per `customer_id` is a quirk of this dataset, not a general rule.
- Orders can have several payments and (rarely) several reviews.

## 3. Six business components
A component is a business responsibility. It can use several tables, and a table can serve more than one component.

In [ ]:
COMPONENTS = {
    "Customer Management": (
        ["customers"], ["orders", "reviews"],
        "Customer identity, location and purchasing history."),
    "Product Management": (
        ["products"], ["order_items", "category_translation"],
        "Catalogue, categories, item prices and product demand."),
    "Seller Management": (
        ["sellers"], ["order_items", "orders"],
        "Seller identity, location and sales contribution."),
    "Order & Sales Management": (
        ["orders", "order_items"], ["customers", "products", "sellers"],
        "Order lifecycle, quantities, status and merchandise sales."),
    "Payment & Revenue": (
        ["payments"], ["orders", "order_items"],
        "Payment methods, installments and values (not the same as merchandise sales)."),
    "Delivery & Customer Experience": (
        ["orders", "reviews"], ["customers", "sellers", "geolocation"],
        "Delivery timestamps and review outcomes."),
}

component_df = pd.DataFrame(
    [(name, ", ".join(p), ", ".join(s), what) for name, (p, s, what) in COMPONENTS.items()],
    columns=["Component", "Primary tables", "Supporting tables", "Responsibility"],
)
display(component_df)

## 4. How the components interact
- **Direct:** a shared key in the data links them.
- **Indirect:** the link can be built by going through another component.
- **Hypothesis:** plausible, but the data alone does not prove it.

In [ ]:
INTERACTIONS = [
    ("Customer Management", "Order & Sales Management", "Direct", "customer_id links customer to order"),
    ("Product Management", "Order & Sales Management", "Direct", "product_id links product to order item"),
    ("Seller Management", "Order & Sales Management", "Direct", "seller_id links seller to order item"),
    ("Order & Sales Management", "Payment & Revenue", "Direct", "order_id links order to payments"),
    ("Order & Sales Management", "Delivery & Customer Experience", "Direct", "order_id links order to timestamps and reviews"),
    ("Product Management", "Payment & Revenue", "Indirect", "item price and quantity next to order payment values"),
    ("Seller Management", "Payment & Revenue", "Indirect", "payments attributed to sellers through order items"),
    ("Seller Management", "Delivery & Customer Experience", "Indirect", "seller items joined to delivery and reviews"),
    ("Delivery & Customer Experience", "Customer Management", "Indirect", "delivery and review history rolled up per customer"),
    ("Customer Management", "Product Management", "Indirect", "purchase history rolled up by product and category"),
    ("Delivery & Customer Experience", "Order & Sales Management", "Hypothesis", "delivery experience may affect future orders"),
    ("Customer Management", "Order & Sales Management", "Hypothesis", "past purchases and satisfaction may predict future buying"),
]

interaction_df = pd.DataFrame(INTERACTIONS, columns=["From", "To", "Type", "Mechanism"])
display(interaction_df)

Solid arrows are direct links, dashed are indirect, and the two "test" arrows are hypotheses.

```mermaid
flowchart TD
    C[Customer Management]
    P[Product Management]
    S[Seller Management]
    O[Order & Sales Management]
    R[Payment & Revenue]
    D[Delivery & Customer Experience]

    C -->|customer_id| O
    P -->|product_id via order items| O
    S -->|seller_id via order items| O
    O -->|order_id| R
    O -->|order_id and timestamps| D

    P -.->|purchase mix and category demand| C
    S -.->|seller performance attribution| D
    D -.->|review and delivery history| C
    S -.->|seller-level transaction attribution| R

    D -.->|repeat-purchase effect: test| O
    C -.->|future purchase propensity: test| O
```

## 5. What is observed and what is not
Keeps the twin honest about what it can claim.

In [ ]:
EVIDENCE = [
    ("Order contains item lines", "Observed", "orders.order_id -> order_items.order_id"),
    ("Item line references a product", "Observed", "order_items.product_id -> products.product_id"),
    ("Item line references a seller", "Observed", "order_items.seller_id -> sellers.seller_id"),
    ("Order has payment records", "Observed", "orders.order_id -> payments.order_id"),
    ("Order has review records", "Observed", "orders.order_id -> reviews.order_id"),
    ("Delivery timestamps are recorded", "Observed", "empty for some statuses, e.g. canceled"),
    ("Delivery delay changes review score", "Association to test", "actual vs estimated delivery against review_score"),
    ("Review score changes repeat purchases", "Association to test", "needs customer-level analysis over time"),
    ("Price changes cause demand changes", "Not identified", "observational data cannot give price elasticity"),
    ("Marketing spend drives acquisition", "Not in the data", "no marketing or channel data in these tables"),
]

evidence_df = pd.DataFrame(EVIDENCE, columns=["Statement", "Status", "Evidence / requirement"])
display(evidence_df)

## 6. Limits of the dataset
Olist supports historical analysis of customers, products, sellers, orders, payments, delivery and reviews.

It does **not** contain marketing spend, inventory or stock-outs, supplier costs or operating expenses, or any experiment data. Anything built on those needs outside data or clearly labelled assumptions.